In [ ]:
# Notebook: Continuous-Time Second-Order Systems - Interactive Step Response & Performance Indices
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interactive
from IPython.display import display

t_vals = np.linspace(0, 20, 800)

def second_order_step_response(t, tau, zeta, kappa):
    tau = max(tau, 1e-3)
    
    if zeta > 1.0:
        omega_d = np.sqrt(zeta**2 - 1.0) / tau
        sigma = zeta / tau
        term = np.exp(-sigma * t) * (np.cosh(omega_d * t) + (zeta / np.sqrt(zeta**2 - 1.0)) * np.sinh(omega_d * t))
        y = kappa * (1.0 - term)
    elif np.isclose(zeta, 1.0, atol=1e-3):
        y = kappa * (1.0 - (1.0 + t / tau) * np.exp(-t / tau))
    else:
        omega_d = np.sqrt(1.0 - zeta**2) / tau
        sigma = zeta / tau
        phi = np.arctan2(np.sqrt(1.0 - zeta**2), zeta)
        y = kappa * (1.0 - (np.exp(-sigma * t) / np.sqrt(1.0 - zeta**2)) * np.sin(omega_d * t + phi))
    return y

# HTML widget για τα performance metrics κάτω από το γράφημα
metrics_widget = widgets.HTML(value="")

def update_second_order_system(tau, zeta, kappa):
    # Δημιουργία χώρου με αρκετό περιθώριο δεξιά για να χωράει το εξωτερικό υπόμνημα
    fig, ax = plt.subplots(figsize=(10, 4.5))
    fig.subplots_adjust(right=0.75)
    
    y_vals = second_order_step_response(t_vals, tau, zeta, kappa)
    
    if zeta > 1.0:
        regime_str = rf"Overdamped ($\zeta$ = {zeta:.2f} > 1)"
        color = 'royalblue'
    elif np.isclose(zeta, 1.0, atol=1e-3):
        regime_str = rf"Critically Damped ($\zeta$ = {zeta:.2f} = 1)"
        color = 'darkcyan'
    else:
        regime_str = rf"Underdamped ($\zeta$ = {zeta:.2f} < 1)"
        color = 'crimson'
        
    ax.plot(t_vals, y_vals, color=color, lw=2.5, label=r'$y(t)$ (Step Response)')
    ax.axhline(kappa, color='gray', linestyle='--', alpha=0.7, label=rf'Steady-state ($\kappa$ = {kappa:.2f})')
    
    # Υπολογισμός Δεικτών Απόδοσης (Performance Indices)
    idx_50 = np.where(y_vals >= 0.5 * kappa)[0]
    tau_d = t_vals[idx_50[0]] if len(idx_50) > 0 else np.nan

    if zeta < 1.0:
        omega_d = np.sqrt(1.0 - zeta**2) / tau
        tau_p = np.pi / omega_d
        peak_val = kappa * (1.0 + np.exp(-zeta * np.pi / np.sqrt(1.0 - zeta**2)))
        mp_percent = (peak_val - kappa) / kappa * 100.0
        idx_peak = np.where(t_vals >= tau_p)[0][0]
        tau_r = t_vals[idx_peak]
        
        if tau_p <= t_vals[-1]:
            ax.scatter([tau_p], [peak_val], color='black', zorder=5)
            ax.annotate(rf'Peak $M_p$ ({peak_val:.2f})', xy=(tau_p, peak_val), 
                        xytext=(tau_p + 0.3, peak_val + 0.05),
                        arrowprops=dict(facecolor='black', shrink=0.05, width=1, headwidth=4), fontsize=8)
    else:
        idx_10 = np.where(y_vals >= 0.1 * kappa)[0]
        idx_90 = np.where(y_vals >= 0.9 * kappa)[0]
        t_10 = t_vals[idx_10[0]] if len(idx_10) > 0 else 0
        t_90 = t_vals[idx_90[0]] if len(idx_90) > 0 else 0
        tau_r = t_90 - t_10
        tau_p = np.nan
        mp_percent = 0.0

    band = 0.02 * kappa
    outside_band = np.where(np.abs(y_vals - kappa) > band)[0]
    tau_s = t_vals[outside_band[-1]] if len(outside_band) > 0 and outside_band[-1] < len(t_vals)-1 else 0.0

    ax.axhline(0, color='black', linewidth=1)
    ax.axvline(0, color='black', linewidth=1)
    ax.set_xlim(0, 20)
    ax.set_ylim(-0.1, max(2.0 * kappa, 2.2))
    ax.set_xlabel(r'Time $t$', fontsize=11)
    ax.set_ylabel(r'Output $y(t)$', fontsize=11)
    ax.set_title(rf'Second-Order Step Response: {regime_str}', fontsize=11)
    ax.grid(True, linestyle=':', alpha=0.6)
    
    # Μεταφορά υπομνήματος εκτός γραφήματος, επάνω δεξιά
    ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1.0), frameon=True, fontsize=9)
    
    plt.show()
    
    # Ενημέρωση των metrics κάτω από το γράφημα
    peak_time_str = f"{tau_p:.3f} s" if not np.isnan(tau_p) else "N/A (Overdamped)"
    metrics_widget.value = f"""
    <div style="font-family: Arial; font-size: 13px; background-color: #f8f9fa; padding: 10px; border-radius: 5px; border: 1px solid #ddd; margin-top: 5px;">
        <b>📊 Performance Indices [{regime_str}]:</b><br>
        &bull; <b>Delay Time ($\tau_d$, 50%):</b> {tau_d:.3f} s &nbsp;|&nbsp; 
        &bull; <b>Rise Time ($\tau_r$):</b> {tau_r:.3f} s &nbsp;|&nbsp; 
        &bull; <b>Peak Time ($\tau_p$):</b> {peak_time_str} <br>
        &bull; <b>Peak Overshoot ($M_p$):</b> {mp_percent:.2f}% &nbsp;|&nbsp; 
        &bull; <b>Settling Time ($\tau_s$, 2%):</b> {tau_s:.3f} s
    </div>
    """

# Ορισμός διαδραστικού πάνελ
interactive_plot = interactive(
    update_second_order_system,
    tau=widgets.FloatSlider(value=1.5, min=0.2, max=5.0, step=0.1, description=r'$\tau$:'),
    zeta=widgets.FloatSlider(value=0.5, min=0.05, max=2.0, step=0.05, description=r'$\zeta$:'),
    kappa=widgets.FloatSlider(value=1.0, min=0.5, max=3.0, step=0.1, description=r'$\kappa$:')
)

display(interactive_plot, metrics_widget)